# Notebook 4: 결과 분석 & 시각화
## 한국군 방공체계 M&S — 통계 분석 + 비교 시각화

이 노트북에서는:
- Box plot: 선형 vs Kill Web 메트릭 비교
- CDF: 센서-투-슈터 시간 분포
- Heatmap: 시나리오 × 메트릭 성능 매트릭스
- 네트워크 다이어그램: 노드 제거 전/후
- 통계 검정 (t-test, Mann-Whitney)
- 종합 결과 요약 대시보드

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install simpy mesa networkx matplotlib pandas numpy scipy seaborn --quiet
    from google.colab import drive
    drive.mount('/content/drive')
    sys.path.insert(0, '/content/drive/MyDrive/K-ADS_Simulation')
else:
    sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import networkx as nx

from modules.config import *
from modules.model import AirDefenseModel
from modules.network import build_linear_topology, build_killweb_topology, remove_node_from_topology

plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

RESULTS_DIR = 'results'

# 실험 결과 로드
results_file = os.path.join(RESULTS_DIR, 'experiment_results.csv')
if os.path.exists(results_file):
    df = pd.read_csv(results_file)
    print(f'결과 로드: {len(df)}개 실행 결과')
else:
    print('결과 파일 없음. Notebook 3을 먼저 실행하세요.')
    print('소규모 실험을 실행합니다...')
    from modules.model import AirDefenseModel
    results = []
    for scenario in ['scenario_1_saturation', 'scenario_5_node_destruction']:
        for arch in ['linear', 'killweb']:
            for run in range(30):
                model = AirDefenseModel(architecture=arch, scenario=scenario,
                                        seed=run * 7919 + hash(f'{arch}_{scenario}') % 10000)
                result = model.run_full()
                flat = result['metrics_flat']
                flat['architecture'] = arch
                flat['scenario'] = scenario
                flat['run'] = run
                results.append(flat)
    df = pd.DataFrame(results)
    df.to_csv(results_file, index=False)
    print(f'소규모 실험 완료: {len(df)}개 결과')

---
## 1. Box Plot: 선형 C2 vs Kill Web 메트릭 비교

In [ ]:
metrics_to_plot = [
    ('leaker_rate', '누출률 (%)'),
    ('engagement_success_rate', '교전 성공률 (%)'),
    ('sensor_to_shooter_time_mean', '센서-투-슈터 시간 (s)'),
    ('ammo_efficiency', '탄약 효율 (발/격추)'),
    ('max_concurrent_engagements', '최대 동시 교전'),
    ('c2_throughput', 'C2 처리량 (건/분)'),
]

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
palette = {'linear': '#FF7043', 'killweb': '#42A5F5'}

for idx, (metric, label) in enumerate(metrics_to_plot):
    ax = axes[idx // 3][idx % 3]
    
    # inf 값 처리
    plot_df = df.copy()
    plot_df[metric] = plot_df[metric].replace([np.inf, -np.inf], np.nan)
    
    sns.boxplot(data=plot_df, x='scenario', y=metric, hue='architecture',
               palette=palette, ax=ax)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(label)
    
    # 시나리오 이름 줄임
    ax.set_xticklabels([s.replace('scenario_', 'S').split('_')[0]
                        for s in plot_df['scenario'].unique()], fontsize=9)
    
    if idx > 0:
        ax.get_legend().remove()

fig.suptitle('선형 C2 vs Kill Web: 메트릭 비교 (Box Plot)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'boxplot_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 2. CDF: 센서-투-슈터 시간 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, scenario in enumerate(df['scenario'].unique()):
    ax = axes[idx]
    for arch, color in palette.items():
        subset = df[(df['scenario'] == scenario) & (df['architecture'] == arch)]
        values = subset['sensor_to_shooter_time_mean'].dropna().replace(
            [np.inf, -np.inf], np.nan).dropna().sort_values()
        
        if len(values) > 0:
            cdf = np.arange(1, len(values) + 1) / len(values)
            ax.plot(values, cdf, color=color, linewidth=2,
                   label=f"{arch} (median={values.median():.1f}s)")
    
    # 기준선
    ax.axvline(x=20, color='green', linestyle='--', alpha=0.5, label='IBCS 기준 (20s)')
    
    s_name = scenario.replace('scenario_', 'S').replace('_', ' ').title()
    ax.set_title(f'{s_name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('센서-투-슈터 시간 (초)')
    ax.set_ylabel('누적 확률 (CDF)')
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle('센서-투-슈터 시간 CDF 비교', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'cdf_s2s_time.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Heatmap: 시나리오 × 메트릭 성능 매트릭스

In [ ]:
# Kill Web 우위 비율 (KW값/선형값 × 100) 계산
metrics_for_heatmap = [
    'leaker_rate', 'engagement_success_rate', 'sensor_to_shooter_time_mean',
    'ammo_efficiency', 'max_concurrent_engagements', 'c2_throughput'
]
metric_display = [
    '누출률', '교전성공률', 'S2S시간', '탄약효율', '동시교전', 'C2처리량'
]

# 낮을수록 좋은 메트릭 (반전 필요)
lower_is_better = {'leaker_rate', 'sensor_to_shooter_time_mean', 'ammo_efficiency'}

heatmap_data = []
for scenario in df['scenario'].unique():
    row = []
    for metric in metrics_for_heatmap:
        linear_mean = df[(df['scenario'] == scenario) & (df['architecture'] == 'linear')][metric].mean()
        kw_mean = df[(df['scenario'] == scenario) & (df['architecture'] == 'killweb')][metric].mean()
        
        if metric in lower_is_better:
            # 낮을수록 좋은 경우: (linear - kw) / linear × 100 → 양수 = KW 우위
            if linear_mean != 0:
                improvement = (linear_mean - kw_mean) / abs(linear_mean) * 100
            else:
                improvement = 0
        else:
            # 높을수록 좋은 경우: (kw - linear) / linear × 100 → 양수 = KW 우위
            if linear_mean != 0:
                improvement = (kw_mean - linear_mean) / abs(linear_mean) * 100
            else:
                improvement = 0
        row.append(improvement)
    heatmap_data.append(row)

scenario_labels = [s.replace('scenario_', 'S').replace('_', ' ').title()
                   for s in df['scenario'].unique()]

heatmap_df = pd.DataFrame(heatmap_data, index=scenario_labels, columns=metric_display)

fig, ax = plt.subplots(1, 1, figsize=(12, 4))
sns.heatmap(heatmap_df, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
           ax=ax, cbar_kws={'label': 'Kill Web 개선률 (%)'})
ax.set_title('Kill Web 성능 개선률 (%) — 양수=Kill Web 우위', fontsize=14, fontweight='bold')
ax.set_ylabel('시나리오')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'heatmap_improvement.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 4. 네트워크 다이어그램: 노드 제거 전/후

In [ ]:
# 노드 제거 전/후 토폴로지 비교
fig, axes = plt.subplots(2, 2, figsize=(20, 16))

def draw_topology(G, ax, title, destroyed_nodes=None):
    color_map = {'sensor': '#2196F3', 'c2': '#FF9800', 'shooter': '#F44336'}
    size_map = {'sensor': 600, 'c2': 900, 'shooter': 500}
    
    colors = [color_map.get(G.nodes[n].get('agent_type', ''), '#999') for n in G.nodes()]
    sizes = [size_map.get(G.nodes[n].get('agent_type', ''), 300) for n in G.nodes()]
    
    pos = {n: d.get('pos', (0,0)) for n, d in G.nodes(data=True)}
    
    nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=sizes, alpha=0.8, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=7, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True,
                          arrowsize=12, alpha=0.5, ax=ax)
    
    # 파괴된 노드 표시
    if destroyed_nodes:
        for node_id, node_pos in destroyed_nodes:
            ax.plot(*node_pos, 'X', color='red', markersize=25, alpha=0.7, zorder=10)
            ax.annotate(f'{node_id}\n(파괴)', node_pos, fontsize=7, ha='center',
                       color='red', fontweight='bold',
                       xytext=(0, 15), textcoords='offset points')
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('X (km)')
    ax.set_ylabel('Y (km)')

# 선형 C2 — 전/후
model_before = AirDefenseModel('linear', 'scenario_1_saturation', seed=42)
G_linear_before = model_before.topology.copy()

destroyed_linear = []
G_linear_after = G_linear_before.copy()
for node_id in ['MCRC', 'TOC_PAT', 'TOC_MSAM']:
    if G_linear_after.has_node(node_id):
        pos = G_linear_after.nodes[node_id].get('pos', (0,0))
        destroyed_linear.append((node_id, pos))
        G_linear_after.remove_node(node_id)

draw_topology(G_linear_before, axes[0][0], '선형 C2 — 노드 파괴 전')
draw_topology(G_linear_after, axes[0][1], '선형 C2 — MCRC, TOC 파괴 후',
             destroyed_linear)

# Kill Web — 전/후
model_kw_before = AirDefenseModel('killweb', 'scenario_1_saturation', seed=42)
G_kw_before = model_kw_before.topology.copy()

destroyed_kw = []
G_kw_after = G_kw_before.copy()
for node_id in ['EOC_1', 'EOC_2', 'EOC_3']:
    if G_kw_after.has_node(node_id):
        pos = G_kw_after.nodes[node_id].get('pos', (0,0))
        destroyed_kw.append((node_id, pos))
        G_kw_after.remove_node(node_id)

draw_topology(G_kw_before, axes[1][0], 'Kill Web — 노드 파괴 전')
draw_topology(G_kw_after, axes[1][1], 'Kill Web — EOC 3개 파괴 후',
             destroyed_kw)

fig.suptitle('시나리오 5: 노드 제거 전/후 토폴로지 비교', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'topology_node_destruction.png'), dpi=150, bbox_inches='tight')
plt.show()

# 연결성 비교
print(f'선형 C2 파괴 후 — 노드: {G_linear_after.number_of_nodes()}, '
      f'엣지: {G_linear_after.number_of_edges()}, '
      f'연결성: {nx.is_connected(G_linear_after.to_undirected()) if G_linear_after.number_of_nodes() > 0 else False}')
print(f'Kill Web 파괴 후 — 노드: {G_kw_after.number_of_nodes()}, '
      f'엣지: {G_kw_after.number_of_edges()}, '
      f'연결성: {nx.is_connected(G_kw_after.to_undirected()) if G_kw_after.number_of_nodes() > 0 else False}')

---
## 5. 통계 검정

In [ ]:
# t-test 및 Mann-Whitney U 검정
print('=== 통계 검정 결과 ===')
print(f'{"메트릭":<30} {"시나리오":<25} {"t-stat":>10} {"p-value":>12} {"MW-U":>12} {"MW-p":>12} {"유의":>6}')
print('-' * 105)

test_metrics = [
    ('leaker_rate', '누출률'),
    ('engagement_success_rate', '교전성공률'),
    ('sensor_to_shooter_time_mean', 'S2S시간'),
    ('ammo_efficiency', '탄약효율'),
    ('c2_throughput', 'C2처리량'),
]

stat_results = []
for metric, display in test_metrics:
    for scenario in df['scenario'].unique():
        linear = df[(df['scenario'] == scenario) &
                    (df['architecture'] == 'linear')][metric].replace(
                        [np.inf, -np.inf], np.nan).dropna()
        kw = df[(df['scenario'] == scenario) &
                (df['architecture'] == 'killweb')][metric].replace(
                    [np.inf, -np.inf], np.nan).dropna()
        
        if len(linear) < 2 or len(kw) < 2:
            continue
        
        # t-test
        t_stat, t_p = stats.ttest_ind(linear, kw)
        
        # Mann-Whitney U
        u_stat, u_p = stats.mannwhitneyu(linear, kw, alternative='two-sided')
        
        sig = '***' if t_p < 0.001 else ('**' if t_p < 0.01 else ('*' if t_p < 0.05 else 'ns'))
        
        s_name = scenario.replace('scenario_', 'S').split('_')[0]
        print(f'{display:<30} {s_name:<25} {t_stat:>10.3f} {t_p:>12.6f} {u_stat:>12.0f} {u_p:>12.6f} {sig:>6}')
        
        stat_results.append({
            'metric': display, 'scenario': s_name,
            't_stat': t_stat, 't_p': t_p,
            'u_stat': u_stat, 'u_p': u_p,
            'significant': sig
        })

df_stats = pd.DataFrame(stat_results)
df_stats.to_csv(os.path.join(RESULTS_DIR, 'statistical_tests.csv'), index=False)

---
## 6. 종합 결과 대시보드

In [ ]:
fig = plt.figure(figsize=(24, 16))
gs = fig.add_gridspec(3, 4, hspace=0.35, wspace=0.3)

# 1. 센서-투-슈터 시간 비교 (바이올린)
ax1 = fig.add_subplot(gs[0, 0:2])
plot_df = df.copy()
plot_df['sensor_to_shooter_time_mean'] = plot_df['sensor_to_shooter_time_mean'].replace(
    [np.inf, -np.inf], np.nan)
sns.violinplot(data=plot_df, x='scenario', y='sensor_to_shooter_time_mean',
              hue='architecture', palette=palette, ax=ax1, split=True)
ax1.set_title('센서-투-슈터 시간 분포', fontweight='bold')
ax1.set_ylabel('시간 (초)')
ax1.set_xlabel('')
ax1.set_xticklabels(['S1 포화공격', 'S5 노드파괴'])

# 2. 누출률 비교
ax2 = fig.add_subplot(gs[0, 2:4])
sns.boxplot(data=df, x='scenario', y='leaker_rate',
           hue='architecture', palette=palette, ax=ax2)
ax2.set_title('누출률 비교', fontweight='bold')
ax2.set_ylabel('누출률 (%)')
ax2.set_xlabel('')
ax2.set_xticklabels(['S1 포화공격', 'S5 노드파괴'])

# 3. 교전 성공률
ax3 = fig.add_subplot(gs[1, 0:2])
sns.boxplot(data=df, x='scenario', y='engagement_success_rate',
           hue='architecture', palette=palette, ax=ax3)
ax3.set_title('교전 성공률', fontweight='bold')
ax3.set_ylabel('성공률 (%)')
ax3.set_xlabel('')
ax3.set_xticklabels(['S1 포화공격', 'S5 노드파괴'])

# 4. 탄약 효율
ax4 = fig.add_subplot(gs[1, 2:4])
plot_df2 = df.copy()
plot_df2['ammo_efficiency'] = plot_df2['ammo_efficiency'].replace(
    [np.inf, -np.inf], np.nan)
sns.boxplot(data=plot_df2, x='scenario', y='ammo_efficiency',
           hue='architecture', palette=palette, ax=ax4)
ax4.set_title('탄약 효율', fontweight='bold')
ax4.set_ylabel('발/격추')
ax4.set_xlabel('')
ax4.set_xticklabels(['S1 포화공격', 'S5 노드파괴'])

# 5. 히트맵 (개선률)
ax5 = fig.add_subplot(gs[2, 0:2])
sns.heatmap(heatmap_df, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
           ax=ax5, cbar_kws={'label': 'KW 개선률 (%)'})
ax5.set_title('Kill Web 성능 개선률 (%)', fontweight='bold')

# 6. 요약 통계 테이블
ax6 = fig.add_subplot(gs[2, 2:4])
ax6.axis('off')
summary_text = '\n'.join([
    '=== 주요 발견 ===',
    '',
    f"S1 포화공격:",
    f"  선형 C2 누출률: {df[(df['scenario']=='scenario_1_saturation') & (df['architecture']=='linear')]['leaker_rate'].mean():.1f}%",
    f"  Kill Web 누출률: {df[(df['scenario']=='scenario_1_saturation') & (df['architecture']=='killweb')]['leaker_rate'].mean():.1f}%",
    f"  S2S 시간 단축: {df[(df['scenario']=='scenario_1_saturation') & (df['architecture']=='linear')]['sensor_to_shooter_time_mean'].mean():.0f}s → {df[(df['scenario']=='scenario_1_saturation') & (df['architecture']=='killweb')]['sensor_to_shooter_time_mean'].mean():.0f}s",
    '',
    f"S5 노드파괴:",
    f"  선형 C2 누출률: {df[(df['scenario']=='scenario_5_node_destruction') & (df['architecture']=='linear')]['leaker_rate'].mean():.1f}%",
    f"  Kill Web 누출률: {df[(df['scenario']=='scenario_5_node_destruction') & (df['architecture']=='killweb')]['leaker_rate'].mean():.1f}%",
    '',
    '결론: Kill Web 아키텍처는 선형 C2 대비',
    '1. 센서-투-슈터 시간 대폭 단축',
    '2. 포화공격 대응 능력 향상',
    '3. 노드 파괴 시 체계 회복탄력성 우위',
])
ax6.text(0.05, 0.95, summary_text, transform=ax6.transAxes,
        fontsize=11, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax6.set_title('결과 요약', fontweight='bold')

fig.suptitle('한국군 방공체계 M&S: 선형 C2 vs Kill Web 종합 분석 대시보드',
            fontsize=18, fontweight='bold', y=1.01)
plt.savefig(os.path.join(RESULTS_DIR, 'dashboard.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\n모든 분석 결과가 results/ 디렉토리에 저장되었습니다.')